FUNCIONANDO

In [ ]:
%pip install mediapipe 
%pip install opencv-python 
%pip install numpy 
%pip install tqdm
%pip install scipy

Imports

In [ ]:
import cv2
import json
import numpy as np
from tqdm import tqdm
from scipy.spatial.transform import Rotation as R

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

Configuração

In [ ]:
VIDEO_PATH = "Video/video.mp4"
MODEL_PATH = "Models/pose_landmarker_lite.task"

BVH_OUTPUT = "bvh/animation_v5.bvh"

In [ ]:
BaseOptions = python.BaseOptions
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
RunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_poses=1
)

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

In [ ]:
BVH_HEADER = """HIERARCHY
ROOT Hips
{
    OFFSET 0 0 0
    CHANNELS 6 Xposition Yposition Zposition Zrotation Xrotation Yrotation

    JOINT Spine1
    {
        OFFSET 0 10 0
        CHANNELS 3 Zrotation Xrotation Yrotation

        JOINT Spine2
        {
            OFFSET 0 10 0
            CHANNELS 3 Zrotation Xrotation Yrotation

            JOINT Spine3
            {
                OFFSET 0 10 0
                CHANNELS 3 Zrotation Xrotation Yrotation

                JOINT Spine4
                {
                    OFFSET 0 10 0
                    CHANNELS 3 Zrotation Xrotation Yrotation
                }
            }
        }
    }

    JOINT LeftUpLeg
    {
        OFFSET 5 0 0
        CHANNELS 3 Zrotation Xrotation Yrotation

        JOINT LeftLeg
        {
            OFFSET 0 -15 0
            CHANNELS 3 Zrotation Xrotation Yrotation
        }
    }

    JOINT RightUpLeg
    {
        OFFSET -5 0 0
        CHANNELS 3 Zrotation Xrotation Yrotation

        JOINT RightLeg
        {
            OFFSET 0 -15 0
            CHANNELS 3 Zrotation Xrotation Yrotation
        }
    }
}
"""

In [ ]:
frames = []

with PoseLandmarker.create_from_options(options) as landmarker:

    for i in tqdm(range(frame_count)):
        ok, frame = cap.read()
        if not ok:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        ts = int(i * 1000 / fps)
        result = landmarker.detect_for_video(mp_image, ts)

        if not result.pose_landmarks:
            continue

        pose = result.pose_landmarks[0]

        # posição base
        lhip = np.array([pose[23].x, pose[23].y, pose[23].z])
        rhip = np.array([pose[24].x, pose[24].y, pose[24].z])

        hip = (lhip + rhip) / 2 * 100

        frame = []

        # ROOT
        frame += [hip[0], hip[1], hip[2], 0.0, 0.0, 0.0]

        # SPINE (SEM rotação)
        frame += [0.0, 0.0, 0.0]
        frame += [0.0, 0.0, 0.0]
        frame += [0.0, 0.0, 0.0]
        frame += [0.0, 0.0, 0.0]

        # LEGS (SEM rotação)
        frame += [0.0, 0.0, 0.0]  # LeftUpLeg
        frame += [0.0, 0.0, 0.0]  # LeftLeg

        frame += [0.0, 0.0, 0.0]  # RightUpLeg
        frame += [0.0, 0.0, 0.0]  # RightLeg

        frames.append(frame)

In [ ]:
motion = []

expected_len = len(frames[0])

for f in frames:
    if len(f) != expected_len:
        continue
    motion.append(" ".join(f"{v:.6f}" for v in f))

with open(BVH_OUTPUT, "w") as f:
    f.write(BVH_HEADER)
    f.write("\nMOTION\n")
    f.write(f"Frames: {len(motion)}\n")
    f.write(f"Frame Time: {1/fps:.6f}\n")

    for line in motion:
        f.write(line + "\n")